# Salidas Novedades BDUA EPS

Son las novedades de regimen especail o inconsistencia de los afiliados, que la EPS reporta ante la ADRES

# Cargar novedades EPS

carga novedades reportadas por la EPS

In [ ]:
from pathlib import Path

import pandas as pd


ruta_ns = Path(
    r"C:\Users\osmarrincon\OneDrive - 891856000_CAPRESOCA E P S"
    r"\Capresoca\AlmostClear\Procesos BDUA\Subsidiados"
    r"\Procesos BDUA EPS\NS\NS validado\All\All-2026.TXT"
)

ruta_nc = Path(
    r"C:\Users\osmarrincon\OneDrive - 891856000_CAPRESOCA E P S"
    r"\Capresoca\AlmostClear\Procesos BDUA\Contributivo"
    r"\Procesos BDUA EPS\NC\NC VAL\All NC VAL\All-2026.TXT"
)

opciones_lectura = {
    "sep": ",",
    "encoding": "cp1252",  # Codificación ANSI habitual en Windows
    "dtype": str,          # Conserva documentos y códigos con ceros iniciales
    "keep_default_na": False,
    "low_memory": False,
}

df_ns = pd.read_csv(ruta_ns, **opciones_lectura)
df_nc = pd.read_csv(ruta_nc, **opciones_lectura)

# Permite identificar el archivo de procedencia de cada registro
df_ns["ORIGEN_ARCHIVO"] = "NS"
df_nc["ORIGEN_ARCHIVO"] = "NC"

df_unificado = pd.concat(
    [df_ns, df_nc],
    ignore_index=True,
    sort=False,
)

print(df_unificado.shape)
print(df_unificado.head())
print(df_unificado["ORIGEN_ARCHIVO"].value_counts())

## Salidas de afiliados N14

Vamos a filtrar los tipos de novedades para dejar solo las novedades de salidas reportadas por la EPS

In [ ]:
df_N14_EPS = df_unificado.loc[
    df_unificado["NOVEDAD"].str.strip().str.upper().eq("N14")
].copy()

print(f"Registros N14 reportados por la EPS: {len(df_N14_EPS):,}")
display(df_N14_EPS.head())

## Fallecimeintos N09 procesos BDUA

In [ ]:
df_N09_EPS = df_unificado.loc[
    df_unificado["NOVEDAD"].str.strip().str.upper().eq("N09")
].copy()

print(f"Registros N09 reportados por la EPS: {len(df_N09_EPS):,}")
display(df_N09_EPS.head())

## Reactivación N31 procesos BDUA

In [ ]:
df_N31_EPS = df_unificado.loc[
    df_unificado["NOVEDAD"].str.strip().str.upper().eq("N31")
].copy()

print(f"Registros N31 reportados por la EPS: {len(df_N31_EPS):,}")
display(df_N31_EPS.head())

# Novedades Municipios
Novedades que reportan las alcaldias al regimen subsidiado

In [ ]:
from pathlib import Path

import pandas as pd


ruta_novedades_alcaldias = Path(
    r"C:\Users\osmarrincon\OneDrive - 891856000_CAPRESOCA E P S"
    r"\Capresoca\AlmostClear\Procesos BDUA\Subsidiados"
    r"\Novedades municipios\All\All-2026.TXT"
)

df_novedades_alcaldias = pd.read_csv(
    ruta_novedades_alcaldias,
    sep=",",
    encoding="cp1252",     # Codificación ANSI de Windows
    header=0,              # La primera fila contiene los encabezados
    dtype=str,             # Conserva documentos y códigos como texto
    keep_default_na=False,
    low_memory=False,
)

# Información de procedencia
df_novedades_alcaldias["ORIGEN_REPORTE"] = "ALCALDÍAS"
df_novedades_alcaldias["REGIMEN"] = "SUBSIDIADO"
df_novedades_alcaldias["PROCESO"] = "NOVEDADES REPORTADAS POR LAS ALCALDÍAS"

print(
    f"Registros cargados: "
    f"{len(df_novedades_alcaldias):,}"
)

print(
    f"Columnas: "
    f"{len(df_novedades_alcaldias.columns)}"
)

display(df_novedades_alcaldias.head())

In [ ]:
# Normalizar la columna NOVEDAD
df_novedades_alcaldias["NOVEDAD"] = (
    df_novedades_alcaldias["NOVEDAD"]
    .astype("string")
    .str.strip()
    .str.upper()
)


# Dejar únicamente fallecimientos y retiros por inconsistencias
df_novedades_alcaldias = (
    df_novedades_alcaldias.loc[
        df_novedades_alcaldias["NOVEDAD"].isin(["N09", "N13"])
    ]
    .copy()
)


# Clasificación general
df_novedades_alcaldias["REGIMEN"] = "SUBSIDIADO"
df_novedades_alcaldias["MOVIMIENTO"] = "SALIDA"
df_novedades_alcaldias["ORIGEN_REPORTE"] = "ALCALDÍAS"
df_novedades_alcaldias["PROCESO"] = (
    "NOVEDADES REPORTADAS POR LAS ALCALDÍAS"
)


# Asignar la causal según la novedad
causales_alcaldias = {
    "N09": "Fallecimiento reportado por la alcaldía",
    "N13": (
        "Retiro por inconsistencias en la afiliación, afiliación en otro "
        "municipio según encuesta Sisbén o inconsistencias en tablas de referencia"
    ),
}

df_novedades_alcaldias["CAUSAL"] = (
    df_novedades_alcaldias["NOVEDAD"]
    .map(causales_alcaldias)
)

In [ ]:
conteo_novedad = (
    df_novedades_alcaldias["NOVEDAD"]
    .value_counts(dropna=False)
    .rename_axis("NOVEDAD")
    .reset_index(name="CANTIDAD")
)

print(conteo_novedad.to_string(index=False))

# Novedades Ministerio de Salud

In [ ]:
from pathlib import Path

import pandas as pd


ruta_ministerio_ns = Path(
    r"C:\Users\osmarrincon\OneDrive - 891856000_CAPRESOCA E P S"
    r"\Capresoca\AlmostClear\Procesos BDUA\Subsidiados"
    r"\RESULTADOS ESPECIALES\VAL\2026"
)

ruta_ministerio_nc = Path(
    r"C:\Users\osmarrincon\OneDrive - 891856000_CAPRESOCA E P S"
    r"\Capresoca\AlmostClear\Procesos BDUA\Contributivo"
    r"\RESULTADOS ESPECIALES\VAL\2026"
)

In [ ]:
columnas_agregadas = {
    "ORIGEN_ARCHIVO",
    "ORIGEN_REPORTE",
    "REGIMEN",
    "MOVIMIENTO",
    "CAUSAL",
    "PROCESO",
    "CODIGO_REGIMEN",
}

columnas_bdua = [
    columna
    for columna in df_unificado.columns
    if columna not in columnas_agregadas
]

print(f"Columnas esperadas: {len(columnas_bdua)}")
columnas_bdua = [
    columna
    for columna in columnas_bdua
    if columna != "Nombre_Archivo"
]
print(columnas_bdua)

In [ ]:
def cargar_archivos_val(carpeta, regimen, codigo_regimen):
    archivos = sorted(carpeta.rglob("*.VAL"))

    if not archivos:
        raise FileNotFoundError(
            f"No se encontraron archivos .VAL en: {carpeta}"
        )

    dataframes = []

    for archivo in archivos:
        df_archivo = pd.read_csv(
            archivo,
            sep=",",
            encoding="cp1252",
            header=None,             # Los archivos no tienen encabezado
            names=columnas_bdua,     # Encabezados del proceso BDUA
            dtype=str,
            keep_default_na=False,
            low_memory=False,
        )

        # Información de trazabilidad
        df_archivo["ARCHIVO_FUENTE"] = archivo.name
        df_archivo["RUTA_ARCHIVO"] = str(archivo)
        df_archivo["CODIGO_REGIMEN"] = codigo_regimen
        df_archivo["REGIMEN"] = regimen
        df_archivo["ORIGEN_REPORTE"] = "MINISTERIO DE SALUD"
        df_archivo["PROCESO"] = (
            "NOVEDADES REPORTADAS POR EL MINISTERIO DE SALUD"
        )

        dataframes.append(df_archivo)

    return pd.concat(
        dataframes,
        ignore_index=True,
        sort=False,
    )

In [ ]:
df_ministerio_ns = cargar_archivos_val(
    carpeta=ruta_ministerio_ns,
    regimen="SUBSIDIADO",
    codigo_regimen="NS",
)

df_ministerio_nc = cargar_archivos_val(
    carpeta=ruta_ministerio_nc,
    regimen="CONTRIBUTIVO",
    codigo_regimen="NC",
)

df_novedades_ministerio = pd.concat(
    [
        df_ministerio_ns,
        df_ministerio_nc,
    ],
    ignore_index=True,
    sort=False,
)

print(
    f"Registros cargados del Ministerio de Salud: "
    f"{len(df_novedades_ministerio):,}"
)

print("\nRegistros por régimen:")
print(df_novedades_ministerio["REGIMEN"].value_counts())

display(df_novedades_ministerio.head())

In [ ]:
# Normalizar el código de novedad
df_novedades_ministerio["NOVEDAD"] = (
    df_novedades_ministerio["NOVEDAD"]
    .astype("string")
    .str.strip()
    .str.upper()
)


# Conservar únicamente N09 y N14
df_novedades_ministerio = (
    df_novedades_ministerio.loc[
        df_novedades_ministerio["NOVEDAD"].isin(["N09", "N14"])
    ]
    .copy()
)


# Clasificar los movimientos
df_novedades_ministerio["MOVIMIENTO"] = "SALIDA"
df_novedades_ministerio["ORIGEN_REPORTE"] = "MINISTERIO DE SALUD"
df_novedades_ministerio["PROCESO"] = (
    "NOVEDADES REPORTADAS POR EL MINISTERIO DE SALUD"
)


# Causales
causales_ministerio = {
    "N09": "Fallecimiento reportado por el Ministerio de Salud",
    "N14": (
        "Paso al régimen especial o retiro por inconsistencias "
        "reportadas por el Ministerio de Salud"
    ),
}

df_novedades_ministerio["CAUSAL"] = (
    df_novedades_ministerio["NOVEDAD"]
    .map(causales_ministerio)
)

In [ ]:
df_novedades_ministerio["NOVEDAD"] = (
    df_novedades_ministerio["NOVEDAD"]
    .astype("string")
    .str.strip()
    .str.upper()
)

resumen_novedades_ministerio = (
    df_novedades_ministerio
    .groupby(
        ["REGIMEN", "NOVEDAD"],
        dropna=False,
    )
    .size()
    .reset_index(name="CANTIDAD")
    .sort_values(
        ["REGIMEN", "NOVEDAD"],
        ignore_index=True,
    )
)

display(resumen_novedades_ministerio)

# Traslados y movilidad Regimen subsidiado procesos BDUA S1

In [ ]:
from pathlib import Path

import pandas as pd


ruta_traslados_movilidad_ns = Path(
    r"C:\Users\osmarrincon\OneDrive - 891856000_CAPRESOCA E P S"
    r"\Capresoca\AlmostClear\Procesos BDUA\Subsidiados"
    r"\Procesos BDUA EPS\Automatico-S1\All\All-2026.TXT"
)


df_traslados_movilidad_subsidiado = pd.read_csv(
    ruta_traslados_movilidad_ns,
    sep=",",
    encoding="cp1252",     # Codificación ANSI de Windows
    header=0,              # Primera fila como encabezado
    dtype=str,             # Conserva documentos y códigos como texto
    keep_default_na=False,
    low_memory=False,
)


# Clasificación general
df_traslados_movilidad_subsidiado["REGIMEN"] = "SUBSIDIADO"
df_traslados_movilidad_subsidiado["MOVIMIENTO"] = "INGRESO"
df_traslados_movilidad_subsidiado["ORIGEN_REPORTE"] = "EPS"
df_traslados_movilidad_subsidiado["PROCESO"] = (
    "TRASLADOS Y MOVILIDADES"
)


print(
    "Registros cargados de traslados y movilidades del subsidiado:",
    f"{len(df_traslados_movilidad_subsidiado):,}",
)

print(
    "Cantidad de columnas:",
    len(df_traslados_movilidad_subsidiado.columns),
)

display(df_traslados_movilidad_subsidiado.head())

In [ ]:
# Normalizar la entidad de origen
entidad_origen = (
    df_traslados_movilidad_subsidiado["ENT_ID_ORIGEN"]
    .astype("string")
    .str.strip()
    .str.upper()
)


# Identificar registros originados en CAPRESOCA
entidades_capresoca = ["EPS025", "EPSC25"]

es_movilidad_reactivacion = entidad_origen.isin(
    entidades_capresoca
)


# Agregar columnas de clasificación
df_traslados_movilidad_subsidiado["REGIMEN"] = "SUBSIDIADO"
df_traslados_movilidad_subsidiado["MOVIMIENTO"] = "INGRESO"
df_traslados_movilidad_subsidiado["ORIGEN_REPORTE"] = "EPS"
df_traslados_movilidad_subsidiado["PROCESO"] = (
    "TRASLADOS Y MOVILIDADES AL RÉGIMEN SUBSIDIADO"
)


# Tipo de ingreso
df_traslados_movilidad_subsidiado["TIPO_INGRESO"] = (
    "TRASLADO DE EPS"
)

df_traslados_movilidad_subsidiado.loc[
    es_movilidad_reactivacion,
    "TIPO_INGRESO",
] = "MOVILIDAD O REACTIVACIÓN"


# Causal
df_traslados_movilidad_subsidiado["CAUSAL"] = (
    "Ingreso al régimen subsidiado por traslado desde otra EPS"
)

df_traslados_movilidad_subsidiado.loc[
    es_movilidad_reactivacion,
    "CAUSAL",
] = (
    "Ingreso al régimen subsidiado por movilidad o reactivación"
)

In [ ]:
entidad_sin_identificar = entidad_origen.isna() | entidad_origen.eq("")

df_traslados_movilidad_subsidiado.loc[
    entidad_sin_identificar,
    "TIPO_INGRESO",
] = "NO IDENTIFICADO"

df_traslados_movilidad_subsidiado.loc[
    entidad_sin_identificar,
    "CAUSAL",
] = "Entidad de origen no informada"

In [ ]:
resumen_traslados_movilidad = (
    df_traslados_movilidad_subsidiado
    .groupby(
        [
            "PROCESO",
            "REGIMEN",
            "MOVIMIENTO",
            "TIPO_INGRESO",
            "CAUSAL",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="CANTIDAD")
)

display(resumen_traslados_movilidad)

In [ ]:
conteo_entidad_origen = (
    df_traslados_movilidad_subsidiado
    .groupby(
        ["ENT_ID_ORIGEN", "TIPO_INGRESO"],
        dropna=False,
    )
    .size()
    .reset_index(name="CANTIDAD")
    .sort_values("CANTIDAD", ascending=False)
)

display(conteo_entidad_origen)

# Traslados y movilidad contributivo R1

In [ ]:
from pathlib import Path

import pandas as pd


ruta_traslados_movilidad_nc = Path(
    r"C:\Users\osmarrincon\OneDrive - 891856000_CAPRESOCA E P S"
    r"\Capresoca\AlmostClear\Procesos BDUA\Contributivo"
    r"\Procesos BDUA EPS\AUTOMATICO R1\All\All-2026.TXT"
)


df_traslados_movilidad_contributivo = pd.read_csv(
    ruta_traslados_movilidad_nc,
    sep=",",
    encoding="cp1252",
    header=0,
    dtype=str,
    keep_default_na=False,
    low_memory=False,
)

print(
    "Registros cargados:",
    f"{len(df_traslados_movilidad_contributivo):,}",
)

display(df_traslados_movilidad_contributivo.head())

In [ ]:
entidad_origen = (
    df_traslados_movilidad_contributivo["ENT_ID_ORIGEN"]
    .astype("string")
    .str.strip()
    .str.upper()
)

entidades_capresoca = ["EPS025", "EPSC25"]

es_movilidad_reactivacion = entidad_origen.isin(
    entidades_capresoca
)

entidad_sin_identificar = (
    entidad_origen.isna()
    | entidad_origen.eq("")
)

In [ ]:
df_traslados_movilidad_contributivo["REGIMEN"] = "CONTRIBUTIVO"
df_traslados_movilidad_contributivo["MOVIMIENTO"] = "INGRESO"
df_traslados_movilidad_contributivo["ORIGEN_REPORTE"] = "EPS"
df_traslados_movilidad_contributivo["PROCESO"] = (
    "TRASLADOS Y MOVILIDADES AL RÉGIMEN CONTRIBUTIVO"
)


# Clasificación inicial: traslado desde otra EPS
df_traslados_movilidad_contributivo["TIPO_INGRESO"] = (
    "TRASLADO DE EPS"
)

df_traslados_movilidad_contributivo["CAUSAL"] = (
    "Ingreso al régimen contributivo por traslado desde otra EPS"
)


# Movilidad o reactivación cuando la entidad de origen es CAPRESOCA
df_traslados_movilidad_contributivo.loc[
    es_movilidad_reactivacion,
    "TIPO_INGRESO",
] = "MOVILIDAD O REACTIVACIÓN"

df_traslados_movilidad_contributivo.loc[
    es_movilidad_reactivacion,
    "CAUSAL",
] = (
    "Ingreso al régimen contributivo por movilidad o reactivación"
)


# Entidades de origen vacías
df_traslados_movilidad_contributivo.loc[
    entidad_sin_identificar,
    "TIPO_INGRESO",
] = "NO IDENTIFICADO"

df_traslados_movilidad_contributivo.loc[
    entidad_sin_identificar,
    "CAUSAL",
] = "Entidad de origen no informada"

# Tabla resumen

## Novedades EPS

In [ ]:
# Crear una copia de cada grupo y clasificar el movimiento
df_N14_resumen = df_N14_EPS.copy()
df_N14_resumen["MOVIMIENTO"] = "SALIDA"
df_N14_resumen["CAUSAL"] = (
    "Paso al régimen especial o retiro por inconsistencias"
)

df_N09_resumen = df_N09_EPS.copy()
df_N09_resumen["MOVIMIENTO"] = "SALIDA"
df_N09_resumen["CAUSAL"] = "Fallecimiento"

df_N31_resumen = df_N31_EPS.copy()
df_N31_resumen["MOVIMIENTO"] = "INGRESO"
df_N31_resumen["CAUSAL"] = (
    "Reactivación por retiro del régimen especial "
    "o corrección de inconsistencias"
)


# Unir los tres tipos de movimientos
df_movimientos_EPS = pd.concat(
    [
        df_N14_resumen,
        df_N09_resumen,
        df_N31_resumen,
    ],
    ignore_index=True,
    sort=False,
)


# Identificar el régimen con los primeros 2 caracteres de Nombre_Archivo
df_movimientos_EPS["CODIGO_REGIMEN"] = (
    df_movimientos_EPS["Nombre_Archivo"]
    .astype("string")
    .str.strip()
    .str[:2]
    .str.upper()
)

df_movimientos_EPS["REGIMEN"] = (
    df_movimientos_EPS["CODIGO_REGIMEN"]
    .map({
        "NS": "SUBSIDIADO",
        "NC": "CONTRIBUTIVO",
    })
    .fillna("NO IDENTIFICADO")
)

df_movimientos_EPS["PROCESO"] = "NOVEDADES REPORTADAS POR LA EPS"

# resumen en formato largo

## novedaes EPS

In [ ]:
cuadro_resumen = (
    df_movimientos_EPS
    .groupby(
        [
            "PROCESO",
            "MOVIMIENTO",
            "NOVEDAD",
            "CAUSAL",
            "REGIMEN",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="CANTIDAD")
    .sort_values(
        ["MOVIMIENTO", "NOVEDAD", "REGIMEN"],
        ascending=[False, True, True],
        ignore_index=True,
    )
)

In [ ]:
# Función para agregar régimen
def agregar_regimen(df):
    df = df.copy()

    codigo_regimen = (
        df["Nombre_Archivo"]
        .astype("string")
        .str.strip()
        .str[:2]
        .str.upper()
    )

    df["REGIMEN"] = (
        codigo_regimen
        .map({
            "NS": "SUBSIDIADO",
            "NC": "CONTRIBUTIVO",
        })
        .fillna("NO IDENTIFICADO")
    )

    return df


# Preparar N14
df_N14_EPS = agregar_regimen(df_N14_EPS)
df_N14_EPS["MOVIMIENTO"] = "SALIDA"
df_N14_EPS["CAUSAL"] = (
    "Paso al régimen especial o retiro por inconsistencias"
)


# Preparar N09
df_N09_EPS = agregar_regimen(df_N09_EPS)
df_N09_EPS["MOVIMIENTO"] = "SALIDA"
df_N09_EPS["CAUSAL"] = "Fallecimiento"


# Preparar N31
df_N31_EPS = agregar_regimen(df_N31_EPS)
df_N31_EPS["MOVIMIENTO"] = "INGRESO"
df_N31_EPS["CAUSAL"] = (
    "Reactivación por retiro del régimen especial "
    "o corrección de inconsistencias"
)


# Unir los tres DataFrames para la hoja de detalle
df_movimientos_EPS = pd.concat(
    [
        df_N14_EPS,
        df_N09_EPS,
        df_N31_EPS,
    ],
    ignore_index=True,
    sort=False,
)

df_movimientos_EPS["PROCESO"] = "NOVEDADES REPORTADAS POR LA EPS"


# Crear tabla resumen
cuadro_resumen = (
    df_movimientos_EPS
    .pivot_table(
        index=[
            "PROCESO",
            "MOVIMIENTO",
            "NOVEDAD",
            "CAUSAL",
        ],
        columns="REGIMEN",
        aggfunc="size",
        fill_value=0,
    )
    .reset_index()
)

cuadro_resumen.columns.name = None

## Novedades alcaldia

In [ ]:
# Crear resumen de las novedades reportadas por alcaldías
resumen_alcaldias = (
    df_novedades_alcaldias
    .groupby(
        [
            "PROCESO",
            "MOVIMIENTO",
            "NOVEDAD",
            "CAUSAL",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="SUBSIDIADO")
)


# Las novedades de alcaldías solamente aplican al régimen subsidiado
resumen_alcaldias["CONTRIBUTIVO"] = 0

resumen_alcaldias["TOTAL"] = (
    resumen_alcaldias["SUBSIDIADO"]
    + resumen_alcaldias["CONTRIBUTIVO"]
)


# Mantener el mismo orden de columnas de cuadro_resumen
resumen_alcaldias = resumen_alcaldias[
    [
        "PROCESO",
        "MOVIMIENTO",
        "NOVEDAD",
        "CAUSAL",
        "SUBSIDIADO",
        "CONTRIBUTIVO",
        "TOTAL",
    ]
]


# Evitar duplicados si esta celda se ejecuta más de una vez
cuadro_resumen = cuadro_resumen.loc[
    cuadro_resumen["PROCESO"].ne(
        "NOVEDADES REPORTADAS POR LAS ALCALDÍAS"
    )
].copy()


# Agregar el resumen de alcaldías al cuadro existente
cuadro_resumen = pd.concat(
    [
        cuadro_resumen,
        resumen_alcaldias,
    ],
    ignore_index=True,
)


# Ordenar el resultado
cuadro_resumen = (
    cuadro_resumen
    .sort_values(
        [
            "PROCESO",
            "MOVIMIENTO",
            "NOVEDAD",
        ],
        ignore_index=True,
    )
)

## Novedades Ministerio de Salud

In [ ]:
# Crear resumen del Ministerio de Salud por régimen
resumen_ministerio = (
    df_novedades_ministerio
    .pivot_table(
        index=[
            "PROCESO",
            "MOVIMIENTO",
            "NOVEDAD",
            "CAUSAL",
        ],
        columns="REGIMEN",
        aggfunc="size",
        fill_value=0,
    )
    .reset_index()
)

resumen_ministerio.columns.name = None


# Garantizar que existan las columnas de ambos regímenes
for regimen in ["SUBSIDIADO", "CONTRIBUTIVO"]:
    if regimen not in resumen_ministerio.columns:
        resumen_ministerio[regimen] = 0


# Calcular el total por causal
resumen_ministerio["TOTAL"] = (
    resumen_ministerio["SUBSIDIADO"]
    + resumen_ministerio["CONTRIBUTIVO"]
)


# Usar el mismo orden de columnas de cuadro_resumen
resumen_ministerio = resumen_ministerio[
    [
        "PROCESO",
        "MOVIMIENTO",
        "NOVEDAD",
        "CAUSAL",
        "SUBSIDIADO",
        "CONTRIBUTIVO",
        "TOTAL",
    ]
]

In [ ]:
proceso_ministerio = (
    "NOVEDADES REPORTADAS POR EL MINISTERIO DE SALUD"
)

# Retirar resultados anteriores del Ministerio para evitar duplicados
cuadro_resumen = cuadro_resumen.loc[
    cuadro_resumen["PROCESO"].ne(proceso_ministerio)
].copy()


# Agregar los nuevos resultados
cuadro_resumen = pd.concat(
    [
        cuadro_resumen,
        resumen_ministerio,
    ],
    ignore_index=True,
)


# Ordenar el cuadro consolidado
cuadro_resumen = (
    cuadro_resumen
    .sort_values(
        [
            "PROCESO",
            "MOVIMIENTO",
            "NOVEDAD",
        ],
        ignore_index=True,
    )
)


display(cuadro_resumen)

## Traslados y movilidades S1

In [ ]:
# Crear resumen de traslados, movilidades y reactivaciones
resumen_traslados_movilidad = (
    df_traslados_movilidad_subsidiado
    .groupby(
        [
            "PROCESO",
            "MOVIMIENTO",
            "TIPO_INGRESO",
            "CAUSAL",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="SUBSIDIADO")
)


resumen_traslados_movilidad["NOVEDAD"] = (
    resumen_traslados_movilidad["TIPO_INGRESO"]
    .replace({
        "TRASLADO DE EPS": "TRASLADO",
        "MOVILIDAD O REACTIVACIÓN": "MOVILIDAD",
        "NO IDENTIFICADO": "NO IDENTIFICADO",
    })
)

# Todos corresponden al régimen subsidiado
resumen_traslados_movilidad["CONTRIBUTIVO"] = 0

resumen_traslados_movilidad["TOTAL"] = (
    resumen_traslados_movilidad["SUBSIDIADO"]
    + resumen_traslados_movilidad["CONTRIBUTIVO"]
)

In [ ]:
resumen_traslados_movilidad = resumen_traslados_movilidad[
    [
        "PROCESO",
        "MOVIMIENTO",
        "NOVEDAD",
        "CAUSAL",
        "SUBSIDIADO",
        "CONTRIBUTIVO",
        "TOTAL",
    ]
]

In [ ]:
proceso_traslados = (
    "TRASLADOS Y MOVILIDADES AL RÉGIMEN SUBSIDIADO"
)

# Eliminar resultados anteriores de este proceso
cuadro_resumen = cuadro_resumen.loc[
    cuadro_resumen["PROCESO"].ne(proceso_traslados)
].copy()


# Agregar el nuevo resumen
cuadro_resumen = pd.concat(
    [
        cuadro_resumen,
        resumen_traslados_movilidad,
    ],
    ignore_index=True,
)


# Ordenar el cuadro consolidado
cuadro_resumen = (
    cuadro_resumen
    .sort_values(
        [
            "PROCESO",
            "MOVIMIENTO",
            "NOVEDAD",
            "CAUSAL",
        ],
        ignore_index=True,
    )
)

display(cuadro_resumen)

In [ ]:
print(
    "Registros del DataFrame:",
    f"{len(df_traslados_movilidad_subsidiado):,}",
)

print(
    "Registros incorporados al resumen:",
    f"{resumen_traslados_movilidad['TOTAL'].sum():,}",
)

In [ ]:
display(cuadro_resumen)

## Traslado  y movilidad R1

In [ ]:
df_traslados_movilidad_contributivo["REGIMEN"] = "CONTRIBUTIVO"
df_traslados_movilidad_contributivo["MOVIMIENTO"] = "INGRESO"
df_traslados_movilidad_contributivo["ORIGEN_REPORTE"] = "EPS"
df_traslados_movilidad_contributivo["PROCESO"] = (
    "TRASLADOS Y MOVILIDADES AL RÉGIMEN CONTRIBUTIVO"
)


# Clasificación inicial: traslado desde otra EPS
df_traslados_movilidad_contributivo["TIPO_INGRESO"] = (
    "TRASLADO DE EPS"
)

df_traslados_movilidad_contributivo["CAUSAL"] = (
    "Ingreso al régimen contributivo por traslado desde otra EPS"
)


# Movilidad o reactivación cuando la entidad de origen es CAPRESOCA
df_traslados_movilidad_contributivo.loc[
    es_movilidad_reactivacion,
    "TIPO_INGRESO",
] = "MOVILIDAD O REACTIVACIÓN"

df_traslados_movilidad_contributivo.loc[
    es_movilidad_reactivacion,
    "CAUSAL",
] = (
    "Ingreso al régimen contributivo por movilidad o reactivación"
)


# Entidades de origen vacías
df_traslados_movilidad_contributivo.loc[
    entidad_sin_identificar,
    "TIPO_INGRESO",
] = "NO IDENTIFICADO"

df_traslados_movilidad_contributivo.loc[
    entidad_sin_identificar,
    "CAUSAL",
] = "Entidad de origen no informada"

In [ ]:
resumen_traslados_movilidad_contributivo = (
    df_traslados_movilidad_contributivo
    .groupby(
        [
            "PROCESO",
            "REGIMEN",
            "MOVIMIENTO",
            "TIPO_INGRESO",
            "CAUSAL",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="CANTIDAD")
)

display(resumen_traslados_movilidad_contributivo)

In [ ]:
# Crear una copia para no modificar el resumen original
resumen_contributivo_para_agregar = (
    resumen_traslados_movilidad_contributivo.copy()
)


# Usar el tipo de ingreso como categoría en la columna NOVEDAD
resumen_contributivo_para_agregar["NOVEDAD"] = (
    resumen_contributivo_para_agregar["TIPO_INGRESO"]
    .replace({
        "TRASLADO DE EPS": "TRASLADO",
        "MOVILIDAD O REACTIVACIÓN": "MOVILIDAD",
        "NO IDENTIFICADO": "NO IDENTIFICADO",
    })
)


# Las cantidades pertenecen al régimen contributivo
resumen_contributivo_para_agregar["SUBSIDIADO"] = 0

resumen_contributivo_para_agregar["CONTRIBUTIVO"] = (
    resumen_contributivo_para_agregar["CANTIDAD"]
)

resumen_contributivo_para_agregar["TOTAL"] = (
    resumen_contributivo_para_agregar["SUBSIDIADO"]
    + resumen_contributivo_para_agregar["CONTRIBUTIVO"]
)


# Dejar la misma estructura de cuadro_resumen
resumen_contributivo_para_agregar = (
    resumen_contributivo_para_agregar[
        [
            "PROCESO",
            "MOVIMIENTO",
            "NOVEDAD",
            "CAUSAL",
            "SUBSIDIADO",
            "CONTRIBUTIVO",
            "TOTAL",
        ]
    ]
)

In [ ]:
cuadro_resumen = pd.concat(
    [
        cuadro_resumen,
        resumen_contributivo_para_agregar,
    ],
    ignore_index=True,
)


# Ordenar el resultado
cuadro_resumen = (
    cuadro_resumen
    .sort_values(
        [
            "PROCESO",
            "MOVIMIENTO",
            "NOVEDAD",
            "CAUSAL",
        ],
        ignore_index=True,
    )
)

display(cuadro_resumen)

# Guardar excel

In [ ]:
from pathlib import Path

import pandas as pd


# Carpeta y archivo de salida
carpeta_salida = Path(
    r"C:\Users\osmarrincon\Documents\Proyectos Python"
    r"\capresoca-data-automation\data"
)

carpeta_salida.mkdir(parents=True, exist_ok=True)

archivo_salida = carpeta_salida / "resumen_ingresos_salidas_EPS.xlsx"



# Garantizar las columnas de ambos regímenes
for regimen in ["SUBSIDIADO", "CONTRIBUTIVO"]:
    if regimen not in cuadro_resumen.columns:
        cuadro_resumen[regimen] = 0


# Calcular total
cuadro_resumen["TOTAL"] = (
    cuadro_resumen["SUBSIDIADO"]
    + cuadro_resumen["CONTRIBUTIVO"]
)

cuadro_resumen = cuadro_resumen[
    [
        "PROCESO",
        "MOVIMIENTO",
        "NOVEDAD",
        "CAUSAL",
        "SUBSIDIADO",
        "CONTRIBUTIVO",
        "TOTAL",
    ]
]


# Guardar ambas tablas en Excel
with pd.ExcelWriter(
    archivo_salida,
    engine="openpyxl",
) as writer:

    cuadro_resumen.to_excel(
        writer,
        sheet_name="Resumen",
        index=False,
    )

    df_movimientos_EPS.to_excel(
        writer,
        sheet_name="Novedades EPS",
        index=False,
    )

    df_novedades_alcaldias.to_excel(
        writer,
        sheet_name="Novedades Alcaldías",
        index=False,
    )

    df_novedades_ministerio.to_excel(
        writer,
        sheet_name="Novedades Alcaldías",
        index=False,
    )

    df_traslados_movilidad_subsidiado.to_excel(
        writer,
        sheet_name="Traslados y Movilidades",
        index=False,
    )


print(f"Archivo guardado correctamente en:\n{archivo_salida}")
print(f"Registros exportados: {len(df_movimientos_EPS):,}")

display(cuadro_resumen)